# Long Context, and Where It Breaks: Lost in the Middle

The runnable code of [Long Context, and Where It Breaks: Lost in the Middle](https://residual-stream.lvtan71.workers.dev/posts/long-context-lost-in-the-middle/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/long-context-lost-in-the-middle.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 80 min end to end.
Downloads Qwen2.5-1.5B-Instruct (about 3 GB).

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "long-context-lost-in-the-middle"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['shared/long-context-lost-in-the-middle/lm.py', 'shared/long-context-lost-in-the-middle/position.py', 'data/blog_train.txt']:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## The experiment

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/long-context-lost-in-the-middle/#the-experiment)

In [ ]:
%%writefile position.py
import json
import random
import sys
import time
import uuid
import torch
from lm import model, tok, DEVICE

# Where in a long prompt does a model find things? Two tasks, each at four
# prompt lengths, with the thing to find at five depths:
#   kv:     a JSON object of random UUID keys and values; give the value for
#           one key. Every other pair is a near-identical distractor
#           (after Liu et al., 2024).
#   needle: one out-of-place sentence hidden in this blog's own prose; give
#           the passphrase it states ("needle in a haystack").
# Qwen2.5-1.5B-Instruct, greedy, chat template; a trial counts as right if
# the answer contains the exact value.
TASK = sys.argv[1]
TRIALS = 12 if TASK == "kv" else 6
SIZES = {"kv": (10, 30, 60, 120), "needle": (750, 2000, 4000, 8000)}[TASK]    # pairs, or prose tokens
SALT = ""
if len(sys.argv) > 2:                     # python position.py kv <sizes> <trials> <salt>: more trials, new draws
    SIZES, TRIALS, SALT = tuple(int(x) for x in sys.argv[2].split(",")), int(sys.argv[3]), sys.argv[4]
DEPTHS = (0.0, 0.25, 0.5, 0.75, 1.0)
blog = open("blog_train.txt").read()
blog_ids = tok(blog).input_ids
m = model()

def ask(content):
    ids = tok.apply_chat_template([{"role": "user", "content": content}], add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True)["input_ids"].to(DEVICE)
    with torch.no_grad():
        out = m.generate(ids, max_new_tokens=60, do_sample=False, repetition_penalty=1.0)
    return ids.shape[1], tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

def kv_trial(n, depth, rng):
    pairs = [(str(uuid.UUID(int=rng.getrandbits(128))), str(uuid.UUID(int=rng.getrandbits(128)))) for _ in range(n)]
    key, value = pairs[round(depth * (n - 1))]
    data = "{" + ", ".join(f'"{k}": "{v}"' for k, v in pairs) + "}"
    return ask(f"Extract the value corresponding to the specified key in the JSON object below. "
               f"Answer with the value only.\n\nJSON data:\n{data}\n\nKey: \"{key}\"\nValue:"), value

def needle_trial(n, depth, rng):
    start = rng.randrange(0, len(blog_ids) - n)
    words = tok.decode(blog_ids[start:start + n]).split(" ")
    secret = f"{rng.choice(['blue', 'red', 'amber', 'grey'])}-heron-{rng.randrange(100, 1000)}"
    words.insert(round(depth * len(words)), f"(The passphrase for the notebook assistant is {secret}.)")
    return ask(f"Read the text below, then answer the question.\n\n{' '.join(words)}\n\n"
               f"What is the passphrase for the notebook assistant? Answer with the passphrase only."), secret

trial = kv_trial if TASK == "kv" else needle_trial
results = {}
t0 = time.perf_counter()
for n in SIZES:
    for depth in DEPTHS:
        rng = random.Random(f"{TASK}-{n}-{depth}{SALT}")
        right, lengths = 0, []
        for _ in range(TRIALS):
            (length, answer), value = trial(n, depth, rng)
            right += value in answer; lengths.append(length)
        results[(n, depth)] = (right, sum(lengths) / len(lengths))
    row = " ".join(f"{results[(n, d)][0]:>2}/{TRIALS}" for d in DEPTHS)
    print(f"{TASK:6s} ~{results[(n, 0.5)][1]:5.0f} prompt tokens: right at depth 0 / 25 / 50 / 75 / 100%: {row}"
          f"   ({time.perf_counter() - t0:.0f} s so far)", flush=True)

In [ ]:
# The post ran both tasks:
%run position.py kv
%run position.py needle

The post printed:

```text
kv     ~  763 prompt tokens: right at depth 0 / 25 / 50 / 75 / 100%: 12/12 12/12 12/12 11/12 12/12   (128 s so far)
kv     ~ 2106 prompt tokens: right at depth 0 / 25 / 50 / 75 / 100%: 12/12 11/12 12/12 12/12 12/12   (340 s so far)
kv     ~ 4106 prompt tokens: right at depth 0 / 25 / 50 / 75 / 100%: 12/12 11/12 10/12  8/12 12/12   (705 s so far)
kv     ~ 8151 prompt tokens: right at depth 0 / 25 / 50 / 75 / 100%: 12/12 10/12  6/12 11/12 12/12   (1411 s so far)
needle ~  822 prompt tokens: right at depth 0 / 25 / 50 / 75 / 100%:  6/6  6/6  6/6  6/6  6/6   (40 s so far)
needle ~ 2072 prompt tokens: right at depth 0 / 25 / 50 / 75 / 100%:  6/6  6/6  6/6  6/6  6/6   (122 s so far)
needle ~ 4072 prompt tokens: right at depth 0 / 25 / 50 / 75 / 100%:  6/6  6/6  6/6  6/6  6/6   (269 s so far)
needle ~ 8072 prompt tokens: right at depth 0 / 25 / 50 / 75 / 100%:  6/6  5/6  6/6  6/6  6/6   (581 s so far)
```

In [ ]:
# 12 more trials at 8,000 tokens (new draws), and 16,000 tokens:
%run position.py kv 120,240 12 -b

The post printed:

```text
kv     ~ 8142 prompt tokens: right at depth 0 / 25 / 50 / 75 / 100%: 12/12  7/12  9/12  9/12 12/12   (936 s so far)
kv     ~16189 prompt tokens: right at depth 0 / 25 / 50 / 75 / 100%: 12/12  2/12 10/12  5/12 12/12   (2448 s so far)
```

In [ ]:
import random
import uuid
import torch
from lm import model, tok, DEVICE

# The 12 key-value objects of position.py's 8,100-token, 50%-depth cell
# (the first run's random draws, regenerated from the same seed), asked two
# ways: as in position.py, and with the key also stated before the data.
# Every answer is classified: right, another pair's value, the right value
# with a character or two wrong, or none of these.
m = model()

def ask(content):
    ids = tok.apply_chat_template([{"role": "user", "content": content}], add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True)["input_ids"].to(DEVICE)
    with torch.no_grad():
        out = m.generate(ids, max_new_tokens=60, do_sample=False, repetition_penalty=1.0)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()

def close(a, b):                              # equal length and at most 2 characters differ
    return len(a) == len(b) and sum(x != y for x, y in zip(a, b)) <= 2

n, depth = 120, 0.5
rng = random.Random(f"kv-{n}-{depth}")
target = round(depth * (n - 1))
counts = {"after only": {}, "before and after": {}}
for trial in range(12):
    pairs = [(str(uuid.UUID(int=rng.getrandbits(128))), str(uuid.UUID(int=rng.getrandbits(128)))) for _ in range(n)]
    key, value = pairs[target]
    data = "{" + ", ".join(f'"{k}": "{v}"' for k, v in pairs) + "}"
    task = "Extract the value corresponding to the specified key in the JSON object below. Answer with the value only."
    for way, content in (("after only", f"{task}\n\nJSON data:\n{data}\n\nKey: \"{key}\"\nValue:"),
                         ("before and after", f"{task}\n\nKey: \"{key}\"\n\nJSON data:\n{data}\n\nKey: \"{key}\"\nValue:")):
        answer = ask(content)
        values = [v for _, v in pairs]
        if value in answer:
            kind = "right"
        elif any(v in answer for v in values):
            kind = f"another pair's value (pair {next(i for i, v in enumerate(values) if v in answer)}, target {target})"
        elif any(close(w.strip('"`.'), value) for w in answer.split()):
            kind = "right value, a character or two wrong"
        else:
            kind = f"none of these: {answer[:40]!r}"
        counts[way][kind] = counts[way].get(kind, 0) + 1
for way, c in counts.items():
    print(f"{way}: " + "; ".join(f"{k}: {v}" for k, v in sorted(c.items(), key=lambda kv: -kv[1])))

The post printed:

```text
after only: right: 6; none of these: '57615999-1111-2222-3333-444444444444': 1; right value, a character or two wrong: 1; another pair's value (pair 40, target 60): 1; another pair's value (pair 55, target 60): 1; another pair's value (pair 52, target 60): 1; another pair's value (pair 50, target 60): 1
before and after: right: 12
```

## Softmax dilution

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/long-context-lost-in-the-middle/#softmax-dilution)

In [ ]:
import math

# Softmax dilution: one relevant key scores delta above n - 1 others that all
# score the same. Its share of the attention is e^delta / (e^delta + n - 1).
for delta in (4, 8):
    shares = ", ".join(f"{n:>7,} keys {math.exp(delta) / (math.exp(delta) + n - 1):6.1%}" for n in (100, 1_000, 10_000, 100_000))
    print(f"score margin {delta}: {shares}")
print(f"margin needed for half the attention among 100,000 keys: ln(99,999) = {math.log(99_999):.1f}")

The post printed:

```text
score margin 4:     100 keys  35.5%,   1,000 keys   5.2%,  10,000 keys   0.5%, 100,000 keys   0.1%
score margin 8:     100 keys  96.8%,   1,000 keys  74.9%,  10,000 keys  23.0%, 100,000 keys   2.9%
margin needed for half the attention among 100,000 keys: ln(99,999) = 11.5
```

## Attention sinks

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/long-context-lost-in-the-middle/#attention-sinks)

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from lm import PATH, tok, DEVICE

# Where does attention go? For 1,024 tokens of this blog, the share of each
# layer's attention (averaged over heads and over every query position past
# the first 64) that lands on the very first token, on the 4 most recent
# tokens, and on everything in between.
# 32-bit floats: with the explicit ("eager") attention needed to read the
# weights out, 16-bit scores overflow in this model.
model = AutoModelForCausalLM.from_pretrained(PATH, dtype=torch.float32, attn_implementation="eager").to(DEVICE).eval()
text = open("blog_train.txt").read()
ids = torch.tensor([tok(text[200_000:206_000]).input_ids[:1024]], device=DEVICE)
with torch.no_grad():
    attn = model(ids, output_attentions=True).attentions          # per layer: (1, heads, T, T)
T = ids.shape[1]
rows = []
for layer, a in enumerate(attn):
    a = a[0, :, 64:, :].float().mean(0)                         # heads averaged; queries 64..T-1
    q = torch.arange(64, T, device=a.device)
    first = a[:, 0].mean().item()
    recent = torch.stack([a[i, max(0, int(qi) - 3):int(qi) + 1].sum() for i, qi in enumerate(q)]).mean().item()
    rows.append((layer, first, recent, 1 - first - recent))
for layer, first, recent, rest in rows:
    if layer % 3 == 0 or layer == len(rows) - 1:
        print(f"layer {layer:2d}: first token {first:5.1%}, last 4 tokens {recent:5.1%}, the other ~1,000 {rest:5.1%}")
print(f"averaged over all {len(rows)} layers: first token {sum(r[1] for r in rows) / len(rows):.1%}, "
      f"last 4 tokens {sum(r[2] for r in rows) / len(rows):.1%}; a uniform split would give the first token "
      f"{sum(1 / (q + 1) for q in range(64, T)) / (T - 64):.2%}")

# Per head rather than per layer (layers 1 to 26, 12 heads each).
per_head = torch.stack([a[0, :, 64:, 0].float().mean(-1) for a in attn[1:27]]).flatten()
print(f"heads in layers 1-26 putting over half their attention on the first token: "
      f"{int((per_head > 0.5).sum())} of {per_head.numel()}; under 5%: {int((per_head < 0.05).sum())}")

# Is it that token, or the first position? Four passages starting with
# different tokens, and the first one again with its first token moved to
# the end: which position gets the most attention, and how much.
def most_attended(ids):
    with torch.no_grad():
        att = model(torch.tensor([ids], device=DEVICE), output_attentions=True).attentions
    share = sum(a[0, :, 64:, :].float().mean((0, 1)) for a in att) / len(att)   # each key's average weight
    v, i = share.max(0)
    return int(i), tok.decode([ids[int(i)]]), v.item()

first = ids[0].tolist()
for name, x in [(f"passage at character {k:,}", tok(text[k:k + 6000]).input_ids[:1024]) for k in (100_000, 200_000, 300_000, 400_000)] \
               + [("first passage, first token moved to the end", first[1:] + first[:1])]:
    pos, token, share = most_attended(x)
    print(f"{name:44s} starts with {tok.decode(x[:1])!r:12s}: most attention on position {pos} ({token!r}), {share:.1%}")

# The digit: the first passage with one token put in front of it.
for lead in ("7", "3", "A", ".", "\n", "the"):
    x = tok(lead).input_ids + first
    pos, token, share = most_attended(x[:1024])
    print(f"first passage with {lead!r:6s} in front: most attention on position {pos} ({token!r}), {share:.1%}")

The post printed:

```text
layer  0: first token  0.0%, last 4 tokens 60.4%, the other ~1,000 39.6%
layer  3: first token 35.1%, last 4 tokens 21.5%, the other ~1,000 43.4%
layer  6: first token 49.5%, last 4 tokens 17.3%, the other ~1,000 33.3%
layer  9: first token 31.7%, last 4 tokens 20.2%, the other ~1,000 48.1%
layer 12: first token 34.9%, last 4 tokens 14.3%, the other ~1,000 50.8%
layer 15: first token 16.8%, last 4 tokens 28.7%, the other ~1,000 54.4%
layer 18: first token 20.2%, last 4 tokens 33.0%, the other ~1,000 46.9%
layer 21: first token 30.6%, last 4 tokens 12.1%, the other ~1,000 57.3%
layer 24: first token 46.8%, last 4 tokens 17.3%, the other ~1,000 36.0%
layer 27: first token  0.1%, last 4 tokens 13.0%, the other ~1,000 86.9%
averaged over all 28 layers: first token 30.5%, last 4 tokens 21.1%; a uniform split would give the first token 0.29%
heads in layers 1-26 putting over half their attention on the first token: 48 of 312; under 5%: 19
passage at character 100,000                 starts with 'rive'      : most attention on position 0 ('rive'), 29.0%
passage at character 200,000                 starts with ' then'     : most attention on position 0 (' then'), 30.5%
passage at character 300,000                 starts with 'Parameter' : most attention on position 0 ('Parameter'), 32.3%
passage at character 400,000                 starts with '7'         : most attention on position 1 ('"'), 30.9%
first passage, first token moved to the end  starts with ' climbs'   : most attention on position 0 (' climbs'), 30.4%
first passage with '7'    in front: most attention on position 1 (' then'), 30.4%
first passage with '3'    in front: most attention on position 1 (' then'), 30.4%
first passage with 'A'    in front: most attention on position 0 ('A'), 30.3%
first passage with '.'    in front: most attention on position 0 ('.'), 30.4%
first passage with '\n'   in front: most attention on position 0 ('\n'), 30.4%
first passage with 'the'  in front: most attention on position 0 ('the'), 30.4%
```